# Ligand preparation for AutoDock Vina

This notebook converts ligands from SDF and SMILES formats into PDBQT format for docking with AutoDock Vina

Pipeline:
- Read input molecules (SDF and SMI)
- Add hydrogens and generate 3D coordinates if needed
- Prepare molecules using Meeko
- Export to PDBQT format

## Libraries

In [37]:
from pathlib import Path

from rdkit import Chem
from rdkit.Chem import AllChem

from meeko import MoleculePreparation, PDBQTWriterLegacy

## Input data

The notebook uses:
- SDF file with ligand structures
- SMI file with SMILES strings

Input files are located in the `input/` directory.

## Functions

In [ ]:
def prepare_molecule_rdkit(mol):
    mol = Chem.Mol(mol)
    mol = Chem.AddHs(mol)

    is_3d = False
    if mol.GetNumConformers() > 0:
        is_3d = mol.GetConformer().Is3D()

    if not is_3d:
        params = AllChem.ETKDGv3()
        params.randomSeed = 42

        status = AllChem.EmbedMolecule(mol, params)
        if status != 0:
            raise ValueError("Failed to generate 3D conformer")

        AllChem.UFFOptimizeMolecule(mol)

    return mol


def write_pdbqt(mol, output_path):
    preparator = MoleculePreparation()
    setups = preparator.prepare(mol)

    if not setups:
        raise ValueError("Meeko failed to prepare molecule")

    pdbqt_string, is_ok, error_msg = PDBQTWriterLegacy.write_string(setups[0])

    if not is_ok:
        raise ValueError(error_msg)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(pdbqt_string)

## Reading SDF file

In [53]:
supplier = Chem.SDMolSupplier("input/example.sdf")
print("Количество молекул:", len(supplier))

Количество молекул: 30


In [17]:
for i, mol in enumerate(supplier):

    if mol is None:
        print(f"Ошибка в молекуле {i}")
        continue

    print("------")
    print("Индекс:", i)

    name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"ligand_{i}"
    print("Имя:", name)

    print("Атомов:", mol.GetNumAtoms())
    print("Конформеров:", mol.GetNumConformers())

    if mol.GetNumConformers() > 0:
        conf = mol.GetConformer()
        print("Is3D:", conf.Is3D())

------
Индекс: 0
Имя: CHEMBL3957375
Атомов: 31
Конформеров: 1
Is3D: False
------
Индекс: 1
Имя: CHEMBL3104445
Атомов: 35
Конформеров: 1
Is3D: False
------
Индекс: 2
Имя: CHEMBL3901969
Атомов: 34
Конформеров: 1
Is3D: False
------
Индекс: 3
Имя: CHEMBL4800145
Атомов: 32
Конформеров: 1
Is3D: False
------
Индекс: 4
Имя: CHEMBL3957572
Атомов: 32
Конформеров: 1
Is3D: False
------
Индекс: 5
Имя: CHEMBL3934158
Атомов: 36
Конформеров: 1
Is3D: False
------
Индекс: 6
Имя: CHEMBL3461502
Атомов: 23
Конформеров: 1
Is3D: False
------
Индекс: 7
Имя: CHEMBL3982666
Атомов: 39
Конформеров: 1
Is3D: False
------
Индекс: 8
Имя: CHEMBL4109555
Атомов: 31
Конформеров: 1
Is3D: False
------
Индекс: 9
Имя: CHEMBL494995
Атомов: 27
Конформеров: 1
Is3D: False
------
Индекс: 10
Имя: CHEMBL3699816
Атомов: 30
Конформеров: 1
Is3D: False
------
Индекс: 11
Имя: CHEMBL1346176
Атомов: 28
Конформеров: 1
Is3D: False
------
Индекс: 12
Имя: CHEMBL104844
Атомов: 26
Конформеров: 1
Is3D: False
------
Индекс: 13
Имя: CHEMBL3976526


## Preparing molecules (RDKit)

In [21]:
supplier = Chem.SDMolSupplier("input/example.sdf", removeHs=False)

prepared_sdf = []

for i, mol in enumerate(supplier, start=1):
    if mol is None:
        print(f"Ошибка чтения молекулы {i}")
        continue

    name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"ligand_{i}"

    try:
        prepared = prepare_molecule_rdkit(mol)
        prepared_sdf.append((name, prepared))
    except Exception as e:
        print(f"Ошибка при подготовке {name}: {e}")

print("Успешно подготовлено молекул:", len(prepared_sdf))

Успешно подготовлено молекул: 30


In [44]:
for name, mol in prepared_sdf:
    print(name)
    print("Атомов:", mol.GetNumAtoms())
    print("Is3D:", mol.GetConformer().Is3D())
    print("-----")

CHEMBL3957375
Атомов: 56
Is3D: True
-----
CHEMBL3104445
Атомов: 57
Is3D: True
-----
CHEMBL3901969
Атомов: 60
Is3D: True
-----
CHEMBL4800145
Атомов: 61
Is3D: True
-----
CHEMBL3957572
Атомов: 57
Is3D: True
-----
CHEMBL3934158
Атомов: 62
Is3D: True
-----
CHEMBL3461502
Атомов: 45
Is3D: True
-----
CHEMBL3982666
Атомов: 61
Is3D: True
-----
CHEMBL4109555
Атомов: 50
Is3D: True
-----
CHEMBL494995
Атомов: 49
Is3D: True
-----
CHEMBL3699816
Атомов: 52
Is3D: True
-----
CHEMBL1346176
Атомов: 58
Is3D: True
-----
CHEMBL104844
Атомов: 50
Is3D: True
-----
CHEMBL3976526
Атомов: 55
Is3D: True
-----
CHEMBL1386741
Атомов: 41
Is3D: True
-----
CHEMBL1644677
Атомов: 51
Is3D: True
-----
CHEMBL4565308
Атомов: 34
Is3D: True
-----
CHEMBL4556129
Атомов: 68
Is3D: True
-----
CHEMBL3922226
Атомов: 55
Is3D: True
-----
CHEMBL352017
Атомов: 51
Is3D: True
-----
CHEMBL1308413
Атомов: 45
Is3D: True
-----
CHEMBL348133
Атомов: 52
Is3D: True
-----
CHEMBL3646377
Атомов: 66
Is3D: True
-----
CHEMBL3649287
Атомов: 72
Is3D: True
--

## Converting to PDBQT (Meeko)

In [26]:
def write_pdbqt(mol, output_path):
    preparator = MoleculePreparation()
    setups = preparator.prepare(mol)

    if not setups:
        raise ValueError("Meeko не смог подготовить молекулу")

    pdbqt_string, is_ok, error_msg = PDBQTWriterLegacy.write_string(setups[0])

    if not is_ok:
        raise ValueError(error_msg)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(pdbqt_string)

In [40]:
out_dir = Path("output")
write_pdbqt(prepared_mol, out_dir / f"{name}.pdbqt")

## Processing SDF ligands

In [41]:
supplier = Chem.SDMolSupplier("input/example.sdf", removeHs=False)

In [42]:
success = 0
failed = 0

for i, mol in enumerate(supplier, start=1):

    if mol is None:
        print(f"Ошибка чтения молекулы {i}")
        failed += 1
        continue

    name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"ligand_{i}"

    try:
        prepared_mol = prepare_molecule_rdkit(mol)

        write_pdbqt(
            prepared_mol,
            Path("output") / f"{name}.pdbqt"
        )

        success += 1
        print(f"Saved: {name}.pdbqt")

    except Exception as e:
        failed += 1
        print(f"Ошибка для {name}: {e}")

Saved: CHEMBL3957375.pdbqt
Saved: CHEMBL3104445.pdbqt
Saved: CHEMBL3901969.pdbqt
Saved: CHEMBL4800145.pdbqt
Saved: CHEMBL3957572.pdbqt
Saved: CHEMBL3934158.pdbqt
Saved: CHEMBL3461502.pdbqt
Saved: CHEMBL3982666.pdbqt
Saved: CHEMBL4109555.pdbqt
Saved: CHEMBL494995.pdbqt
Saved: CHEMBL3699816.pdbqt
Saved: CHEMBL1346176.pdbqt
Saved: CHEMBL104844.pdbqt
Saved: CHEMBL3976526.pdbqt
Saved: CHEMBL1386741.pdbqt
Saved: CHEMBL1644677.pdbqt
Saved: CHEMBL4565308.pdbqt
Saved: CHEMBL4556129.pdbqt
Saved: CHEMBL3922226.pdbqt
Saved: CHEMBL352017.pdbqt
Saved: CHEMBL1308413.pdbqt
Saved: CHEMBL348133.pdbqt
Saved: CHEMBL3646377.pdbqt
Saved: CHEMBL3649287.pdbqt
Saved: CHEMBL3646372.pdbqt
Saved: CHEMBL3461559.pdbqt
Saved: CHEMBL3986453.pdbqt
Saved: CHEMBL3946996.pdbqt
Saved: CHEMBL3699817.pdbqt
Saved: CHEMBL3894759.pdbqt


## Processing SMI ligands

SMILES input does not contain 3D coordinates, so 3D conformers are generated using RDKit before conversion to PDBQT

In [47]:
with open("input/example.smi", "r") as f:
    for _ in range(5):
        print(f.readline().strip())

smiles name
CC(=O)N1C[C@@H]2OCC(=O)N(Cc3ccc(F)cc3)[C@@H]2C1 CHEMBL4959907
Cc1nn(C)cc1-c1cc(C(N)=O)nc2cc(CN3C[C@@H](C)O[C@@H](C(F)(F)F)C3)ccc12 CHEMBL4108673
Cc1nsc(-c2cc(C(N)=O)nc3cc(CN4C[C@@H](C)O[C@@H](C(F)(F)F)C4)ccc23)n1 CHEMBL4107154
Fc1ccc(CN2CCOC(Cn3cncn3)C2)c2ncccc12 CHEMBL3460453


In [51]:
smi_molecules = []

with open("input/example.smi", "r") as f:
    next(f)  # пропускаем заголовок: "smiles name"

    for i, line in enumerate(f, start=1):
        line = line.strip()

        if not line:
            continue

        parts = line.split()

        smiles = parts[0]
        name = parts[1] if len(parts) > 1 else f"ligand_{i}"

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            print(f"Ошибка SMILES: {smiles}")
            continue

        smi_molecules.append((name, mol))

print("Количество молекул:", len(smi_molecules))

Количество молекул: 30


In [59]:
smi_success = 0
smi_failed = 0

for name, mol in smi_molecules:
    try:
        prepared_mol = prepare_molecule_rdkit(mol)

        write_pdbqt(
            prepared_mol,
            Path("output") / f"smi_{name}.pdbqt"
        )

        smi_success += 1

    except Exception as e:
        smi_failed += 1
        print(f"SMI error for {name}: {e}")

print("------")
print("SMI success:", smi_success)
print("SMI failed:", smi_failed)

[RDKit] ERROR:[00:04:41] UFFTYPER: Unrecognized charge state for atom: 27
[RDKit] ERROR:[00:04:41] UFFTYPER: Unrecognized charge state for atom: 27


------
SMI success: 30
SMI failed: 0


функции:

In [45]:
def safe_name(name):
    return re.sub(r"[^\w\-.]+", "_", name)


def prepare_molecule_rdkit(mol):
    mol = Chem.Mol(mol)
    mol = Chem.AddHs(mol)

    is_3d = False
    if mol.GetNumConformers() > 0:
        is_3d = mol.GetConformer().Is3D()

    if not is_3d:
        params = AllChem.ETKDGv3()
        params.randomSeed = 42

        status = AllChem.EmbedMolecule(mol, params)
        if status != 0:
            raise ValueError("Failed to generate 3D conformer")

        AllChem.UFFOptimizeMolecule(mol)

    return mol


def write_pdbqt(mol, output_path):
    preparator = MoleculePreparation()
    setups = preparator.prepare(mol)

    if not setups:
        raise ValueError("Meeko failed to prepare molecule")

    pdbqt_string, is_ok, error_msg = PDBQTWriterLegacy.write_string(setups[0])

    if not is_ok:
        raise ValueError(error_msg)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(pdbqt_string)